# Entity Masking

If model is not downloaded, run
`python -m spacy download en_core_web_sm)`

In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [2]:
# Define the masking rules (mapping specific NER labels to generic tokens)
MASKING_MAP = {
    "ORG": "[ORGANIZATION]",
    "PART_ID": "[PART_NUMBER]",
    "FAILURE_MODE": "[FAILURE_MODE]",
    "DURATION": "[DURATION]"
}

In [3]:
def apply_entity_masking(doc, masking_map):
    """
    Applies entity masking to a spaCy Doc object.
    Replaces entity text with a generic token from MASKING_MAP.
    """
    masked_text = list(doc.text)
    
    # Process entities in reverse order to preserve character indices
    for ent in reversed(doc.ents):
        if ent.label_ in masking_map:
            mask_token = masking_map[ent.label_]
            
            # Replace the entity text with the generic token
            start, end = ent.start_char, ent.end_char
            masked_text[start:end] = list(mask_token)
            
    return "".join(masked_text).replace("  ", " ") # Clean up potential extra spaces

In [4]:
report_input = "The failure was observed on the AX-430 part from Acme Corp, occurring at 100 hours."
doc = nlp(report_input)

## NER Inference

In [5]:
# Override doc.ents with some custom/target entities for a better example
# (In a real scenario, this would come from your custom NER model)
doc.ents = (
    spacy.tokens.Span(doc, 6, 8, label="PART_ID"),     # AX-430 part
    spacy.tokens.Span(doc, 9, 11, label="ORG"),        # Acme Corp
    spacy.tokens.Span(doc, 13, 15, label="DURATION")   # 100 hours
)

In [6]:
masked_report = apply_entity_masking(doc, MASKING_MAP)

print(f"Original Input: {report_input}")
print(f"Masked Input:   {masked_report}")

Original Input: The failure was observed on the AX-430 part from Acme Corp, occurring at 100 hours.
Masked Input:   The failure was observed on the [PART_NUMBER] from [ORGANIZATION], occurring [DURATION] hours.


> This 'Masked Input' is what we will feed to the fine-tuned Mistral-7B.